# はじめに

ボートレースのレーティングをいくつか作成しており、その中の1つのレーティングモデルのコードをまとめておく。


## モデル

ボートレーサーの競走能力を複数の要素に分解し、過去の着順から選手の潜在的な実力を推定するベイズPlackett–Luceモデルである。将来の着順予測ではなく、進入コースやモーターなどの条件を考慮した、公正な選手評価を主目的としている。

### 潜在スコア

レース $r$、日付 $t$、進入コース $c$ における潜在スコアを次のように定義する。

$$
\eta_{r,c,t}
=R_{i,t}+C_{c,t}+RC_{i,c,t}
+VC_{v,c,t}+M_{m,t}+u_{r,c}
$$

- $R_{i,t}$：選手 $i$ の基本実力
- $C_{c,t}$：全国共通のコース効果
- $RC_{i,c,t}$：選手固有のコース適性
- $VC_{v,c,t}$：レース場 $v$ 固有のコース効果
- $M_{m,t}$：モーター $m$ の性能効果
- $u_{r,c}$：レース固有の艇間連動を表す潜在変動

コース効果は基本実力との重複を抑えるため、6コースの合計がゼロになるよう中心化する。

$$
\sum_{c=1}^{6} C_{c,t}=0,\quad
\sum_{c=1}^{6}RC_{i,c,t}=0,\quad
\sum_{c=1}^{6}VC_{v,c,t}=0
$$

### 尤度

6艇の実際の着順を $\pi=(\pi_1,\ldots,\pi_6)$ とすると、Plackett–Luceモデルによる尤度は、

$$
P(\pi\mid\boldsymbol{\eta})
=
\prod_{k=1}^{5}
\frac{\exp(\eta_{\pi_k})}
{\sum_{j=k}^{6}\exp(\eta_{\pi_j})}
$$

### レース内の連動構造

艇間の展開による連動を表現するため、6次元多変量正規分布を導入する。

$$
\boldsymbol z_r\sim N_6(\boldsymbol0,I)\\

\boldsymbol u_r
=
P(\sigma_u L_Q\boldsymbol z_r)\\

P=I-\frac16\boldsymbol1\boldsymbol1^\top\\
$$

ここで $L_QL_Q^\top=Q$、$\sigma_u=0.20$ とし、相関行列は、

$$
Q_{ij}=0.35^{|i-j|}
$$

に固定する。生成した変動を中心化することでレース内の合計をゼロとし、艇同士の連動性を潜在スコアに反映させる。

### 時間発展と事後推定

各パラメータは日次で変化するランダムウォークに従う。

$$
\theta_t\mid\theta_{t-1}
\sim N(\theta_{t-1},\tau_\theta^2)
$$

前日までの事後分布を翌日の事前分布に継承し、経過日数 $\Delta t$ に応じて分散を増加させる。
$$
\mu_t^-=\mu_{t-1}^+
$$

$$
(\sigma_t^-)^2
=(\sigma_{t-1}^+)^2+\tau_\theta^2\Delta t
$$

推定にはMAP推定とLaplace近似を採用する(MCMCは重いので)。まず事後分布の最頻値を求め、その周辺の曲率から正規近似を構築する。
$$
p(\boldsymbol\theta\mid D_{1:t})
\approx N(\hat{\boldsymbol\theta}_{MAP},H^{-1})
$$

得られたMAP推定値と周辺標準偏差をStateに保存し、翌日の推定へ引き継ぐ。ただし、現在の実装ではパラメータ間の事後共分散を保持せず、周辺分散のみを継承する近似を採用している。
この仕組みにより、選手の実力水準とその推定不確実性を分けて表現し、競走条件を考慮しながら日次でレーティングを更新する。

## 設定

In [8]:
from __future__ import annotations

from copy import deepcopy
from datetime import date
from pathlib import Path

import jax
import numpy as np
import polars as pl

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from jax import random
from jax.scipy.special import logsumexp
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoLaplaceApproximation
from numpyro.optim import Adam

Plackett–Luce尤度、MAP推定、Hessianを使ったLaplace近似などを計算するので、JAXで64ビット浮動小数点数を使用できるようにしている。`Q_RHO` は、コース間相関の減衰を定める値で、下記のような相関行列となる。隣り合うコースほど強い正の相関を持ち、離れるほど相関が小さくなる構造。

$$
Q=
\begin{pmatrix}
1&.35&.1225&.0429&.0150&.0053\\
.35&1&.35&.1225&.0429&.0150\\
.1225&.35&1&.35&.1225&.0429\\
.0429&.1225&.35&1&.35&.1225\\
.0150&.0429&.1225&.35&1&.35\\
.0053&.0150&.0429&.1225&.35&1
\end{pmatrix}
$$


In [9]:
# ============================================================
# 1. Settings
# ============================================================
ROOT = Path.cwd().resolve()
RUN_START_DATE = date(2025, 10, 1)
RUN_END_DATE = date(2026, 10, 9)
SAVE_OUTPUT = True

# マッピングデータのカラム名
COL_RACE_DATE = "race_date"
COL_RACE_ID = "race_id"
COL_VENUE = "race_stadium_number"
COL_RACE_NUMBER = "race_number"
COL_BOAT = "racer_boat_number"
COL_RACER = "racer_number"
COL_MOTOR = "racer_assigned_motor_number"
COL_MOTOR_GENERATION = "motor_generation"
COL_COURSE = "racer_course_number"
COL_PLACE = "racer_place_number"

# 初期事前分布の設定
INITIAL_R_MU, INITIAL_R_SIGMA = 0.0, 1.0
INITIAL_C_MU, INITIAL_C_SIGMA = 0.0, 0.5
INITIAL_RC_MU, INITIAL_RC_SIGMA = 0.0, 0.35
INITIAL_VC_MU, INITIAL_VC_SIGMA = 0.0, 0.25
INITIAL_M_MU, INITIAL_M_SIGMA = 0.0, 0.4

# レース固有の変動と艇間相関
SIGMA_U_FIXED = 0.20
TAU_R, TAU_C, TAU_RC, TAU_VC, TAU_M = 0.02, 0.002, 0.008, 0.003, 0.01
Q_RHO = 0.35
Q_FIXED = np.fromfunction(lambda i, j: Q_RHO ** np.abs(i - j), (6, 6))
Q_CHOL = jnp.asarray(np.linalg.cholesky(Q_FIXED), dtype=jnp.float64)
SVI_STEPS = 3000
LEARNING_RATE = 0.01
POSTERIOR_SAMPLES = 300
RANDOM_SEED = 42
MIN_SIGMA = 1e-6

`load_sources()` は、3つのParquetファイルを読み込み、番組表にモーター世代を付与したうえで、推定対象期間の番組表とレース結果を返す関数。番組表、レース結果のデータでは、後続の計算ができないようなデータの不備が発生してレコードはレース単位で除外されている。

In [10]:
# ============================================================
# 2. Source data and data quality
# ============================================================
def load_sources(root: Path):
    generations = pl.read_parquet(root / "motor_generations.parquet")
    programs = pl.read_parquet(root / "train_programs_preprocess.parquet")
    results = pl.read_parquet(root / "common_results_preprocess.parquet")
    programs = programs.join_where(
        generations,
        pl.col(COL_VENUE) == pl.col("race_stadium_number_valid"),
        pl.col(COL_RACE_DATE) >= pl.col("motor_valid_from"),
        pl.col(COL_RACE_DATE) <= pl.col("motor_valid_end"),
        how="left",
    )
    active = programs.filter(pl.col(COL_RACE_DATE).is_between(RUN_START_DATE, RUN_END_DATE, closed="both"))
    return active, results


# ============================================================
# 2. Prep motor_generations
# ============================================================
# programs_all = pl.read_parquet(Path.cwd().resolve() / "train_programs_preprocess.parquet")
# # 1. レース場・日付単位で集計
# daily = (
#     programs_all.group_by(["race_stadium_number", "race_date"])
#     .agg(
#         pl.col("racer_assigned_motor_top_2_percent").max().alias("m2p_max"),
#         pl.col("racer_assigned_motor_top_3_percent").max().alias("m3p_max"),
#     )
#     .sort(["race_stadium_number", "race_date"])
# )


# # 2. モーター交換日を検出
# changes = (
#     daily.with_columns(pl.col("m2p_max").shift(1).over("race_stadium_number").alias("prev_m2p"))
#     .filter((pl.col("m2p_max") == 0) & (pl.col("m3p_max") == 0) & (pl.col("prev_m2p") > 0))
#     .select(
#         "race_stadium_number",
#         pl.col("race_date").alias("motor_valid_from"),
#     )
# )

# # 3. 各レース場の最初の観測日を取得
# initial = daily.group_by("race_stadium_number").agg(pl.col("race_date").min().alias("motor_valid_from"))

# # 4. 初期世代と交換日を結合し、期間を生成
# motor_generations = (
#     pl.concat([initial, changes])
#     .unique(subset=["race_stadium_number", "motor_valid_from"])
#     .sort(["race_stadium_number", "motor_valid_from"])
#     .with_columns((pl.col("motor_valid_from").shift(-1).over("race_stadium_number") - pl.duration(days=1)).alias("motor_valid_end"))
#     .with_columns(pl.col("motor_valid_end").fill_null(date(2050, 12, 31)))
#     .with_columns((pl.col("motor_valid_from").cum_count().over("race_stadium_number") - 1).alias("motor_generation"))
#     .select(
#         pl.col("race_stadium_number").alias("race_stadium_number_valid"),
#         "motor_valid_from",
#         "motor_valid_end",
#         "motor_generation",
#     )
# )

# motor_generations
# motor_generations.write_parquet(Path.cwd().resolve() / "motor_generations.parquet")


`load_day()` は、前の `load_sources()` で読み込んだデータから、指定した1日分の番組表と着順結果を結合し、モデルに渡すためのデータを作成する関数。特に重要なのは、最後の `.sort(["race_id", "place"])` で、Plackett–Luceモデルでは、行の並び順そのものが着順情報になるのでその処理が入っている。

In [11]:
def load_day(programs_source: pl.DataFrame, results_source: pl.DataFrame, race_date: date):
    programs_day = (
        programs_source.filter(pl.col(COL_RACE_DATE) == race_date)
        .select([COL_RACE_ID, COL_RACE_NUMBER, COL_RACER, COL_BOAT, COL_VENUE, COL_MOTOR, COL_MOTOR_GENERATION])
        .rename(
            {
                COL_RACE_ID: "race_id",
                COL_RACE_NUMBER: "race_number",
                COL_RACER: "racer_number",
                COL_BOAT: "boat_number",
                COL_VENUE: "venue",
                COL_MOTOR: "motor",
                COL_MOTOR_GENERATION: "motor_generation",
            }
        )
    )
    results_day = (
        results_source.select([COL_RACE_ID, COL_RACER, COL_COURSE, COL_PLACE])
        .rename({COL_RACE_ID: "race_id", COL_RACER: "racer_number", COL_COURSE: "course", COL_PLACE: "place"})
        .with_columns(pl.col("course").cast(pl.Int32, strict=False), pl.col("place").cast(pl.Int32, strict=False))
    )
    df = programs_day.join(results_day, on=["race_id", "racer_number"], how="inner").sort(["race_id", "place"])

    return df


前日までのレース結果から得られた各パラメータの事後分布を記録した状態を辞書で管理。これを参照して初期→事後＝事前→事後、、、と繰り返す。`advance_state()`が担当しており、前日の事後分布を当日の事前分布に変換、というかDeepCoopする。半年間、毎日Stateを更新するのであれば、`days` は基本的に常に1になり、選手がその日に出走していても、出走していなくても、登録済みの全選手に時間発展が適用される。レースがあっても、体調、体重変化など様々な要素が存在するのでそのあたりの処理。

In [12]:
# ============================================================
# 3. State: keyed by original IDs, never by temporary index
# ============================================================
def fresh_racer():
    return {
        "r_mu": INITIAL_R_MU,
        "r_sigma": INITIAL_R_SIGMA,
        "rc_raw_mu": np.full(6, INITIAL_RC_MU, dtype=np.float64),
        "rc_raw_sigma": np.full(6, INITIAL_RC_SIGMA, dtype=np.float64),
        "last_race_date": None,
    }


def fresh_venue():
    return {"vc_raw_mu": np.full(6, INITIAL_VC_MU, dtype=np.float64), "vc_raw_sigma": np.full(6, INITIAL_VC_SIGMA, dtype=np.float64)}


def fresh_motor():
    return {"mu": INITIAL_M_MU, "sigma": INITIAL_M_SIGMA, "last_race_date": None}


def initial_state():
    return {
        "racers": {},
        "venues": {},
        "motors": {},
        "c_raw_mu": np.full(6, INITIAL_C_MU, dtype=np.float64),
        "c_raw_sigma": np.full(6, INITIAL_C_SIGMA, dtype=np.float64),
        "date": None,
    }


def advance_state(state, today):
    nxt = deepcopy(state)
    if state["date"] is None:
        return nxt
    days = (today - state["date"]).days
    if days <= 0:
        raise ValueError(f"invalid date update: {state['date']} -> {today}")
    for x in nxt["racers"].values():
        x["r_sigma"] = float(np.hypot(x["r_sigma"], TAU_R * np.sqrt(days)))
        x["rc_raw_sigma"] = np.sqrt(x["rc_raw_sigma"] ** 2 + TAU_RC**2 * days)
    for x in nxt["venues"].values():
        x["vc_raw_sigma"] = np.sqrt(x["vc_raw_sigma"] ** 2 + TAU_VC**2 * days)
    for x in nxt["motors"].values():
        x["sigma"] = float(np.hypot(x["sigma"], TAU_M * np.sqrt(days)))
    nxt["c_raw_sigma"] = np.sqrt(nxt["c_raw_sigma"] ** 2 + TAU_C**2 * days)
    return nxt

ここはすごいややこしい。識別子の情報が複数出てくるのと、同じ選手でも毎日識別子が変わるので、不変の選手Noをキーに適切に更新できるようにしているが、イメージこんな感じ。ChatGPTに作成してもらったけど、ややこしい部分の対応関係や流れは理解できる。細かい部分はコードで確認するほうがよい。

<div align='center'><img src='./Boatracerating01.png' width='1200'></div>


`prepare_daily_model_input()` は、その日のレースに登場する選手・レース場・モーターに、「一時的なパラメータ番号」を割り当て、前日までのStateから対応する事前分布を取り出して、モデルへ渡す入力データを作成する関数。処理は大きく4段階に分かれ、

1. Lookupを作る：その日登場する選手・レース場・モーターを抽出し、それぞれに0始まりの番号を割り当てる。
2. Stateを準備する：初登場の選手などには初期事前分布を登録し、過去に登場したものは保存済みの分布を利用する。
3. 着順順のインデックスを作る：各レースの1着から6着までについて、どのパラメータ番号を参照するかを並べる。
4. モデルの入力を作る：パラメータの参照番号と、事前分布の平均・標準偏差を kwargs にまとめる。

| 種類 | 例：選手4321 | 用途 |
| --- | --- | --- |
| 元の識別子 | 4321 | Stateへの保存・読み出し |
| パラメータ位置 | 2 | モデル内部の R[2] を参照 |
| 着順位置 | 0（1着） | PL尤度で1着のスコアとして扱う |

選手4321が1着でも、パラメータ位置は2、この2つの位置をつなぐのが、`racer_param_idx`。
`df_day` に次の結果が入っている。選手4321が1着、レース場は12号場、モーター世代は2。`load_day()` の最後で着順順に並べているので、この順番で渡される。

| 着順 | 選手番号 | 進入コース | モーター |
| --- | --- | --- | --- |
| 1 | 4321 | 2 | 42 |
| 2 | 2345 | 1 | 21 |
| 3 | 3456 | 3 | 13 |
| 4 | 5678 | 4 | 56 |
| 5 | 4567 | 5 | 32 |
| 6 | 6789 | 6 | 18 |

`racer_lookup`で「4321」が「位置2」になる。この番号は選手のレーティングでも着順でもなく、その日のモデル内部で使用する配列の位置。モータ、レース場も同じ。

| racer_param_idx | racer_number |
| --- | --- |
| 0 | 2345 |
| 1 | 3456 |
| 2 | 4321 |
| 3 | 4567 |
| 4 | 5678 |
| 5 | 6789 |

「4321」が使用している「モーター42」の「パラメータ位置は4」となる。モーターはレース場、世代、モータ番号で管理される。

| motor_param_idx | venue | generation | motor |
| --- | --- | --- | --- |
| 0 | 12 | 2 | 13 |
| 1 | 12 | 2 | 18 |
| 2 | 12 | 2 | 21 |
| 3 | 12 | 2 | 32 |
| 4 | 12 | 2 | 42 |
| 5 | 12 | 2 | 56 |

IDを保存し、Stateと対応させるのが次のステップ。これらのリストは、パラメータ位置から元の識別子へ戻すためにも使われる。

```python
racer_ids = [2345, 3456, 4321, 4567, 5678, 6789]
venue_ids = [12]
motor_keys = [
    (12, 2, 13),
    (12, 2, 18),
    (12, 2, 21),
    (12, 2, 32),
    (12, 2, 42),
    (12, 2, 56),
]
```

`rank_rows`で「着順」と「パラメータ位置」をつなぐ。そのために3つのルックアップが使用される。「1着」の「4321」をモデルで評価するときは、「Rの位置2」、「Cの位置1」、「VCのレース場位置0」、「Mの位置4」を参照する。

| 着順 | 選手番号 | racer_param_idx | course_param_idx | venue_param_idx | motor_param_idx |
| --- | --- | --- | --- | --- | --- |
| 1 | 4321 | 2 | 1 | 0 | 4 |
| 2 | 2345 | 0 | 0 | 0 | 2 |
| 3 | 3456 | 1 | 2 | 0 | 0 |
| 4 | 5678 | 4 | 3 | 0 | 5 |
| 5 | 4567 | 3 | 4 | 0 | 3 |
| 6 | 6789 | 5 | 5 | 0 | 1 |


`indexed()`はJAX用に2次元配列にする関数で、レースごとの実際の着順順で下記のように成る。レースごとに`rank_rows`を1行づつ取ったようなイメージで、これが1日のレース分作成される。

```python
racer_param_idx  = [[2, 0, 1, 4, 3, 5]]
course_param_idx = [[1, 0, 2, 3, 4, 5]]
venue_param_idx  = [[0, 0, 0, 0, 0, 0]]
motor_param_idx  = [[4, 2, 0, 5, 3, 1]]
```

`State`の各選手のパラメタは`Lookup`で決めた`racer_ids`の順。

```
r_states = [state["racers"][rid] for rid in racer_ids]
```

この並びにおいて、`prior()`は、`State`から事前分布を取り出す関数で、「4321」は「位置2」の「0.80」となる。この `r_mu` は着順順ではなくパラメータ位置順。

```python
racer_ids = 
[2345, 3456, 4321, 4567, 5678, 6789]
↓
prior(r_states, "r_mu")
↓
r_mu = 
[0.30, 0.10, 0.80, -0.20, 0.50, 0.00]
```

`kwargs`はモデルにわたす引数をまとめたもの。

| 引数 | 配列の形 | 意味 |
| --- | --- | --- |
| racer_param_idx | レース数 × 6 | 着順ごとの選手パラメータ位置 |
| course_param_idx | レース数 × 6 | 着順ごとのコース位置 |
| venue_param_idx | レース数 × 6 | 着順ごとのレース場位置 |
| motor_param_idx | レース数 × 6 | 着順ごとのモーター位置 |
| r_mu, r_sigma | 選手数 |
| c_mu, c_sigma | 6 |
| rc_mu, rc_sigma | 選手数 × 6 |
| vc_mu, vc_sigma | レース場数 × 6 |
| m_mu, m_sigma | モーター数 |

最後にモデル内部で2種類の配列がつなげる。

```python
R = numpyro.sample("R",dist.Normal(r_mu, r_sigma).to_event(1))
↓
R = [R0, R1, R2, R3, R4, R5] なので　racer_ids[2] == 4321、R[2] は4321の実力を表す
```

戻り値が`kwargs`だけではないのはモデルで推定した結果を、元のStateへ書き戻すため。

```
R[2] = 0.85
↓
rid = racer_ids[2]
↓
state["racers"][rid]["r_mu"] = 0.85
```

次の日には出走する選手が変わるので、4321のパラメータ位置が2のままとは限らず、Stateは元の選手番号で保存し、毎日新しいLookupを作成するという設計。

| 処理 | データの並び | 4321の例 |
| --- | --- | --- |
| df_day | 着順順 | 先頭行（1着） |
| racer_lookup | 選手番号順 | パラメータ位置2 |
| state["racers"] | 元の選手番号をキーに管理 | state["racers"][4321] |
| r_states | Lookup順 | リスト位置2 |
| r_mu | Lookup順 | r_mu[2] = 0.80 |
| racer_param_idx | 着順順 | 先頭要素が2 |
| R[racer_param_idx] | 着順順 | 先頭要素が R[2] |
| 推定結果の保存 | 元の選手番号へ戻す | state["racers"][4321] |

この関数は**「元のID → 一時的な位置 → モデルで推定 → 元のIDへ戻す」ための橋渡し**をしている。最大のポイントは、`r_mu` などの事前分布はパラメータ位置順、`racer_param_idx` などの観測用配列は着順順ということ。

In [13]:
# ============================================================
# 4. Single source of truth for IDs, indices AND priors
# ============================================================
def prepare_daily_model_input(df_day: pl.DataFrame, state: dict):
    # lookup は1日のデータが対象で、各カラムの昇順に行番号をふる
    racer_lookup = df_day.select("racer_number").unique().sort("racer_number").with_row_index("racer_param_idx")
    venue_lookup = df_day.select("venue").unique().sort("venue").with_row_index("venue_param_idx")
    motor_cols = ["venue", "motor_generation", "motor"]
    motor_lookup = df_day.select(motor_cols).unique().sort(motor_cols).with_row_index("motor_param_idx")
    racer_ids = racer_lookup["racer_number"].to_list()
    venue_ids = venue_lookup["venue"].to_list()
    motor_keys = list(motor_lookup.select(motor_cols).iter_rows())

    for rid in racer_ids:
        state["racers"].setdefault(rid, fresh_racer())
    for vid in venue_ids:
        state["venues"].setdefault(vid, fresh_venue())
    for key in motor_keys:
        state["motors"].setdefault(key, fresh_motor())

    rank_rows = (
        df_day.join(racer_lookup, on="racer_number", how="left", validate="m:1")
        .join(venue_lookup, on="venue", how="left", validate="m:1")
        .join(motor_lookup, on=motor_cols, how="left", validate="m:1")
        .with_columns((pl.col("course") - 1).cast(pl.Int32).alias("course_param_idx"))
        .sort(["race_id", "place"])
    )

    def _indexed(col):
        return jnp.asarray(rank_rows[col].to_numpy().reshape(-1, 6), dtype=jnp.int32)

    def _prior(items, field):
        return jnp.asarray(np.asarray([x[field] for x in items], dtype=np.float64))

    r_states = [state["racers"][rid] for rid in racer_ids]
    v_states = [state["venues"][vid] for vid in venue_ids]
    m_states = [state["motors"][key] for key in motor_keys]
    kwargs = {
        "racer_param_idx": _indexed("racer_param_idx"),
        "course_param_idx": _indexed("course_param_idx"),
        "venue_param_idx": _indexed("venue_param_idx"),
        "motor_param_idx": _indexed("motor_param_idx"),
        "r_mu": _prior(r_states, "r_mu"),
        "r_sigma": _prior(r_states, "r_sigma"),
        "c_mu": jnp.asarray(state["c_raw_mu"]),
        "c_sigma": jnp.asarray(state["c_raw_sigma"]),
        "rc_mu": _prior(r_states, "rc_raw_mu"),
        "rc_sigma": _prior(r_states, "rc_raw_sigma"),
        "vc_mu": _prior(v_states, "vc_raw_mu"),
        "vc_sigma": _prior(v_states, "vc_raw_sigma"),
        "m_mu": _prior(m_states, "mu"),
        "m_sigma": _prior(m_states, "sigma"),
    }

    return {
        "kwargs": kwargs,
        "racer_ids": racer_ids,
        "venue_ids": venue_ids,
        "motor_keys": motor_keys,
    }

$u_{r,c}$はレース固有のランダム効果。実際のレースでは、1コースが流れたり、3コースがまくったりすることで、他の艇にも影響が及ぶ。そのため、レースごとに6つの変動を考える。

$$
\boldsymbol{u}_r=
\begin{pmatrix}
u_{r,1}\\
u_{r,2}\\
u_{r,3}\\
u_{r,4}\\
u_{r,5}\\
u_{r,6}
\end{pmatrix}
$$

これらは独立ではなく実際のボートレースのように関係＝展開をもたせたい。そのため、相関を持たせるために多変量正規分布を使用する。

$$
\boldsymbol{u}_r
\sim
N_6(\boldsymbol{0},\Sigma_u)
$$

コードでは次のようにすることで、Qが6艇の連動の仕方を決め、SIGMA_U_FIXED で変動の大きさをコントロールする。

$$
\Sigma_u=0.20^2 Q
$$

2艇の場合で件が得るとわかりよい。コレスキー分解$Q=LL^\top$することで、下三角行列が得られ、

$$
Q=
\begin{pmatrix}
1&0.35\\
0.35&1
\end{pmatrix},

L=
\begin{pmatrix}
1&0\\
0.35&\sqrt{1-0.35^2}
\end{pmatrix}
$$

と計算でき、$z_1,z_2\sim N(0,1)$から2×2の艇番ごとの関係を表現する行列を得る。


$$
\begin{pmatrix}u_1\\u_2\end{pmatrix}
=
0.20L
\begin{pmatrix}z_1\\z_2\end{pmatrix}
$$

実際は独立な標準正規変数から6×6の行列となる。

中心化の部分について、`rc_mu`を例にすると、これは引数として渡されるので逆の流れでまとめると、

```python
# 選手ごとに6コース分のパラメータを持っている
RC = RC_raw - RC_raw.mean(axis=1, keepdims=True)
↓
# 引数として、rc_muは渡される
RC_raw = numpyro.sample("RC_raw", dist.Normal(rc_mu, rc_sigma).to_event(2))
↓
state["racers"] = {
    2345: {"rc_raw_mu": np.array([0.20, 0.10, 0.00, -0.10, -0.10, -0.10])},
    3456: {"rc_raw_mu": np.array([-0.10, 0.20, 0.10, 0.00, -0.10, -0.10])},
    4321: {"rc_raw_mu": np.array([0.10, 0.30, -0.10, 0.00, -0.20, -0.10])},
    ...
}
↓
def prior(items, field):
        return jnp.asarray(np.asarray([x[field] for x in items], dtype=np.float64))
rc_mu = prior(items=r_states, field="rc_raw_mu")
rc_mu = [x["rc_raw_mu"] for x in r_states]
↓
rc_mu = jnp.array([
    [ 0.20,  0.10,  0.00, -0.10, -0.10, -0.10],
    [-0.10,  0.20,  0.10,  0.00, -0.10, -0.10],
    [ 0.10,  0.30, -0.10,  0.00, -0.20, -0.10], #4321
    ...
])
```

In [14]:
# ============================================================
# 5. M5 Model
# ============================================================
def pl_loglik(eta):
    loglik = jnp.zeros(eta.shape[0], dtype=eta.dtype)
    for k in range(5):
        loglik = loglik + eta[:, k] - logsumexp(eta[:, k:], axis=1)
    return jnp.sum(loglik)


# m5に意味はなく、5個目のモデルということ
def m5_model(
    racer_param_idx, course_param_idx, venue_param_idx, motor_param_idx, r_mu, r_sigma, c_mu, c_sigma, rc_mu, rc_sigma, vc_mu, vc_sigma, m_mu, m_sigma
):
    R = numpyro.sample("R", dist.Normal(r_mu, r_sigma).to_event(1))
    C_raw = numpyro.sample("C_raw", dist.Normal(c_mu, c_sigma).to_event(1))
    # 当日出走する全選手について、6コース分の潜在パラメータを同時に推定対象
    RC_raw = numpyro.sample("RC_raw", dist.Normal(rc_mu, rc_sigma).to_event(2))
    VC_raw = numpyro.sample("VC_raw", dist.Normal(vc_mu, vc_sigma).to_event(2))
    M = numpyro.sample("M", dist.Normal(m_mu, m_sigma).to_event(1))
    C = C_raw - C_raw.mean()
    # 選手ごとに6コース分のパラメータを持っている
    RC = RC_raw - RC_raw.mean(axis=1, keepdims=True)
    VC = VC_raw - VC_raw.mean(axis=1, keepdims=True)
    n_races = racer_param_idx.shape[0]
    z = numpyro.sample("race_z", dist.Normal(0.0, 1.0).expand((n_races, 6)).to_event(2))
    u_raw = SIGMA_U_FIXED * (z @ Q_CHOL.T)
    u_centered = u_raw - u_raw.mean(axis=1, keepdims=True)
    u_rank = jnp.take_along_axis(u_centered, course_param_idx, axis=1)

    # Model
    eta = (
        R[racer_param_idx]
        + C[course_param_idx]
        + RC[racer_param_idx, course_param_idx]
        + VC[venue_param_idx, course_param_idx]
        + M[motor_param_idx]
        + u_rank
    )
    numpyro.factor("plackett_luce", pl_loglik(eta))


`fit_one_day()` は、昨日までの`State`を事前分布として使い、今日のレース結果からモデルの事後分布を推定し、その結果を元の選手番号・レース場番号・モーターキーに対応させて`State`へ書き戻す関数。
推定結果も `racer_lookup` で決めた`「パラメータ位置順」に得られる。つまり、`R[2] = 0.85` は、「選手4321」のMAP推定値。　`r_mu`に絞って流れを確認する。

```python
map_site = guide.median(result.params)
↓
map_site = {
    "R": [...],
    "C_raw": [...],
    "RC_raw": [[...], ...],
    "VC_raw": [[...], ...],
    "M": [...],
    "race_z": [[...], ...],
}
↓
# racer_ids = [2345, 3456, 4321, 4567, 5678, 6789] の順
map_site["R"]=[0.35, 0.08, 0.85, -0.12, 0.48, 0.03]
↓
map_site["R"][2] = 0.85
↓
r_mu, r_sigma = get_mean_sd("R") # [#0, #1, 0.85, #3. #4, #5] ,[#0, #1, 0.19, #3. #4, #5]
↓
# racer_ids = [2345, 3456, 4321, 4567, 5678, 6789]
for i, rid in enumerate(batch["racer_ids"]):
    posterior["racers"][rid] = {
        "r_mu": float(r_mu[i]),
        # racer_ids = 4321のときは、i=2なので、r_mu[2]となり 0.85 が記録される
        ~~~
    }
```

`prepare_daily_model_input()` が「元のID → パラメータ位置」を担当し、`fit_one_day()` の後半が「パラメータ位置 → 元のID」を担当しています。

| 段階 | データの管理方法 |
| --- | --- |
| 前日のState | 元の選手番号で管理 |
| racer_ids | 元の番号と一時位置の対応表 |
| kwargs["r_mu"] | パラメータ位置順の事前分布 |
| kwargs["racer_param_idx"] | 着順順の参照インデックス |
| map_site["R"] | パラメータ位置順のMAP推定値 |
| draws["R"] | パラメータ位置順の事後サンプル |
| get_mean_sd("R") | パラメータ位置順のμ・σ |
| posterior["racers"] | 元の選手番号へ戻して保存 |


ここまで説明した処理を繋げて実行することで、1日分の更新を完了する。

In [15]:
# ============================================================
# 6. Daily MAP/Laplace fit and keyed state writeback
# ============================================================
def fit_one_day(programs_source, results_source, race_date, state, seed):
    prior = advance_state(state, race_date)
    df_day = load_day(programs_source, results_source, race_date)
    batch = prepare_daily_model_input(df_day, prior)
    kwargs = batch["kwargs"]
    guide = AutoLaplaceApproximation(m5_model)
    svi = SVI(m5_model, guide, Adam(step_size=LEARNING_RATE), Trace_ELBO())
    result = svi.run(random.key(seed), SVI_STEPS, **kwargs)
    map_site = guide.median(result.params)
    draws = guide.sample_posterior(random.key(seed + 1), result.params, sample_shape=(POSTERIOR_SAMPLES,), **kwargs)

    def _get_mean_sd(name):
        mean = np.asarray(map_site[name], dtype=np.float64)
        sd = np.asarray(draws[name], dtype=np.float64).std(axis=0, ddof=1)
        sd = np.maximum(sd, MIN_SIGMA)
        if not np.all(np.isfinite(mean)) or not np.all(np.isfinite(sd)):
            raise ValueError(f"Non-finite Laplace posterior: {name}")
        return mean, sd

    r_mu, r_sigma = _get_mean_sd("R")
    c_raw_mu, c_raw_sigma = _get_mean_sd("C_raw")
    rc_raw_mu, rc_raw_sigma = _get_mean_sd("RC_raw")
    vc_raw_mu, vc_raw_sigma = _get_mean_sd("VC_raw")
    m_mu, m_sigma = _get_mean_sd("M")

    posterior = deepcopy(prior)
    posterior["date"] = race_date
    posterior["c_raw_mu"] = c_raw_mu.copy()
    posterior["c_raw_sigma"] = c_raw_sigma.copy()
    for i, rid in enumerate(batch["racer_ids"]):
        posterior["racers"][rid] = {
            "r_mu": float(r_mu[i]),
            "r_sigma": float(r_sigma[i]),
            "rc_raw_mu": rc_raw_mu[i].copy(),
            "rc_raw_sigma": rc_raw_sigma[i].copy(),
            "last_race_date": race_date,
        }
    for i, vid in enumerate(batch["venue_ids"]):
        posterior["venues"][vid] = {"vc_raw_mu": vc_raw_mu[i].copy(), "vc_raw_sigma": vc_raw_sigma[i].copy()}
    for i, key in enumerate(batch["motor_keys"]):
        posterior["motors"][key] = {"mu": float(m_mu[i]), "sigma": float(m_sigma[i]), "last_race_date": race_date}

    return posterior


In [16]:
# ============================================================
# 7. Run and export
# ============================================================
def write_ratings(root, ranking, course_rating, racer_course_rating, venue_course_rating):
    ranking.write_json(root / "outout.json")
    course_rating.write_json(root / "outout_course.json")
    racer_course_rating.write_json(root / "outout_racer_course.json")
    venue_course_rating.write_json(root / "outout_venue_course.json")


def run():
    programs_run, results_all = load_sources(ROOT)
    run_dates = programs_run.select(COL_RACE_DATE).unique().sort(COL_RACE_DATE).get_column(COL_RACE_DATE).to_list()
    state = initial_state()
    # No initial historic warm start: RUN_START_DATE is the cold-start date.
    for i, race_date in enumerate(run_dates):
        print(f"{i + 1}/{len(run_dates)} {race_date}", flush=True)
        state = fit_one_day(programs_run, results_all, race_date, state, RANDOM_SEED + i)

    # 以降は書き出しのための関数
    ranking = pl.DataFrame(
        {
            "racer_number": list(state["racers"].keys()),
            "mu": [x["r_mu"] for x in state["racers"].values()],
            "sigma": [x["r_sigma"] for x in state["racers"].values()],
            "last_race_date": [x["last_race_date"] for x in state["racers"].values()],
        }
    ).sort("mu", descending=True)

    # C / RC / VC はコース方向で中心化した値が eta に入る。sigma は保存している raw の事後標準偏差。
    courses = np.arange(1, 7)
    course_rating = pl.DataFrame(
        {
            "course": courses,
            "mu": state["c_raw_mu"] - state["c_raw_mu"].mean(),
            "sigma": state["c_raw_sigma"],
        }
    )
    racer_course_rating = pl.DataFrame(
        [
            {
                "racer_number": rid,
                "course": course,
                "mu": float(x["rc_raw_mu"][course - 1] - x["rc_raw_mu"].mean()),
                "sigma": float(x["rc_raw_sigma"][course - 1]),
            }
            for rid, x in state["racers"].items()
            for course in courses
        ]
    )
    venue_course_rating = pl.DataFrame(
        [
            {
                "venue": vid,
                "course": course,
                "mu": float(x["vc_raw_mu"][course - 1] - x["vc_raw_mu"].mean()),
                "sigma": float(x["vc_raw_sigma"][course - 1]),
            }
            for vid, x in state["venues"].items()
            for course in courses
        ]
    )
    if SAVE_OUTPUT:
        write_ratings(ROOT, ranking, course_rating, racer_course_rating, venue_course_rating)
    return state, ranking, course_rating, racer_course_rating, venue_course_rating


In [17]:
SAVE_OUTPUT = True

if __name__ == "__main__":
    final_state, ranking_final, course_rating, racer_course_rating, venue_course_rating = run()

1/374 2025-10-01


100%|██████████| 3000/3000 [00:00<00:00, 3702.04it/s, init loss: 21836.0112, avg. loss [2851-3000]: 1328.9406]


2/374 2025-10-02


100%|██████████| 3000/3000 [00:00<00:00, 3842.33it/s, init loss: 23826.8271, avg. loss [2851-3000]: 1300.5829]


3/374 2025-10-03


...

100%|██████████| 3000/3000 [00:00<00:00, 3577.11it/s, init loss: 142223.2820, avg. loss [2851-3000]: -1430.0248]


373/374 2026-10-08


100%|██████████| 3000/3000 [00:00<00:00, 3419.99it/s, init loss: 139872.4677, avg. loss [2851-3000]: -1348.6911]


374/374 2026-10-09


100%|██████████| 3000/3000 [00:00<00:00, 3196.90it/s, init loss: 129053.5847, avg. loss [2851-3000]: -1363.3761]


最終的に直近1年のレース結果を用いて、選手のランキングを作成するとこうなる。上から、峰選手、白井選手、山田選手、久田選手、池田選手、、、、となる。

In [18]:
import json

ranking = pl.read_json(ROOT / "outout.json").sort("mu", descending=True).head(30).with_row_index("rank", offset=1)

with pl.Config(set_tbl_rows=-1):
    print(ranking.head(30))

shape: (30, 5)
┌──────┬──────────────┬──────────┬──────────┬────────────────┐
│ rank ┆ racer_number ┆ mu       ┆ sigma    ┆ last_race_date │
│ ---  ┆ ---          ┆ ---      ┆ ---      ┆ ---            │
│ u32  ┆ i64          ┆ f64      ┆ f64      ┆ str            │
╞══════╪══════════════╪══════════╪══════════╪════════════════╡
│ 1    ┆ 4320         ┆ 2.042609 ┆ 0.194655 ┆ 2026-10-03     │
│ 2    ┆ 3897         ┆ 1.821573 ┆ 0.193652 ┆ 2026-10-09     │
│ 3    ┆ 4500         ┆ 1.818522 ┆ 0.160659 ┆ 2026-10-09     │
│ 4    ┆ 4188         ┆ 1.714499 ┆ 0.184785 ┆ 2026-10-09     │
│ 5    ┆ 3941         ┆ 1.655449 ┆ 0.178291 ┆ 2026-10-09     │
│ 6    ┆ 4444         ┆ 1.592188 ┆ 0.20936  ┆ 2026-09-30     │
│ 7    ┆ 4205         ┆ 1.588413 ┆ 0.148965 ┆ 2026-10-03     │
│ 8    ┆ 4350         ┆ 1.576403 ┆ 0.168607 ┆ 2026-10-05     │
│ 9    ┆ 3898         ┆ 1.535665 ┆ 0.164792 ┆ 2026-10-02     │
│ 10   ┆ 4238         ┆ 1.528511 ┆ 0.194609 ┆ 2026-10-09     │
│ 11   ┆ 4851         ┆ 1.528323 ┆ 0.174